# 16 - Splunk Fraud Detection

## Введение

**Splunk** - это платформа для сбора, индексации и анализа машинных данных (логов, метрик, событий). Она широко используется в SecOps, DevOps и fraud detection благодаря мощному языку запросов **SPL** (Splunk Processing Language) и возможности обрабатывать терабайты данных в реальном времени.

### Почему Splunk для обнаружения мошенничества?

- **Скорость**: индексированные данные ищутся за миллисекунды
- **Корреляция**: можно объединять события из разных источников (логи авторизации, платежи, геолокация)
- **Real-time alerts**: правила срабатывают сразу при поступлении события
- **ML Toolkit**: встроенная интеграция с scikit-learn для ML-моделей

### Идея блокнота

Мы построим упрощённую модель fraud detection в стиле Splunk:

1. Сгенерируем синтетические логи транзакций
2. Применим **rule-based** правила (пороговые значения, частота, страна)
3. Обучим **ML-модель** (RandomForest) на этих данных
4. Завернём всё в **FastAPI**-сервис с ngrok-туннелем
5. Добавим **интерактивные виджеты** для настройки порогов и параметров модели

> **Важно**: этот ноутбук - учебная имитация Splunk-подхода, а не реальная установка Splunk.

## Настройка среды

Установим все необходимые библиотеки. **Shift+Enter** - запуск ячейки в Colab.

In [ ]:
# Установка библиотек
!pip install pandas numpy matplotlib seaborn scikit-learn fastapi uvicorn pyngrok pydantic ipywidgets -q

In [ ]:
# Импорты и проверка версий
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import time
import threading
import warnings
from datetime import datetime, timedelta
from collections import defaultdict, Counter
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
np.random.seed(42)

print('Pandas:', pd.__version__)
print('NumPy:', np.__version__)
import sklearn; print('Sklearn:', sklearn.__version__)
import fastapi; print('FastAPI:', fastapi.__version__)
print('Среда готова к работе.')

## Генерация синтетических логов

Создадим реалистичный датасет транзакций в стиле Splunk-логов. Поля:

- `timestamp` - время события
- `user_id` - идентификатор пользователя
- `ip` - IP-адрес
- `amount` - сумма транзакции
- `country` - страна
- `event_type` - тип события (login, payment, transfer, refund)
- `is_fraud` - целевая метка (1 - мошенничество)

Мошеннические транзакции составят ~5%, что близко к реальным показателям.

In [ ]:
# Генерация синтетических логов транзакций

def generate_logs(n_normal=10000, n_fraud=500):
    # Количество нормальных и мошеннических транзакций
    n_total = n_normal + n_fraud
    start_time = datetime(2025, 1, 1)
    
    # Список стран с весами (чаще всего транзакции из US/RU/CN)
    countries = ['US', 'RU', 'CN', 'DE', 'FR', 'UK', 'BR', 'IN', 'JP', 'Unknown']
    country_weights = [0.30, 0.20, 0.15, 0.08, 0.07, 0.06, 0.05, 0.04, 0.03, 0.02]
    
    # Типы событий
    event_types = ['login', 'payment', 'transfer', 'refund']
    
    # Пользователи
    user_ids = [f'user_{i:04d}' for i in range(1, 1001)]
    
    # IP-адреса (псевдо)
    def gen_ip():
        return f"{np.random.randint(1, 255)}.{np.random.randint(0, 255)}.{np.random.randint(0, 255)}.{np.random.randint(1, 255)}"
    
    records = []
    
    # --- Нормальные транзакции ---
    for _ in range(n_normal):
        ts = start_time + timedelta(seconds=np.random.randint(0, 30*24*3600))
        records.append({
            'timestamp': ts,
            'user_id': np.random.choice(user_ids),
            'ip': gen_ip(),
            'amount': float(np.round(np.random.exponential(200) + 10, 2)),
            'country': np.random.choice(countries, p=country_weights),
            'event_type': np.random.choice(event_types, p=[0.4, 0.35, 0.20, 0.05]),
            'is_fraud': 0
        })
    
    # --- Мошеннические транзакции ---
    fraud_patterns = ['high_amount', 'rare_country', 'rapid_burst', 'night_event']
    for _ in range(n_fraud):
        ts = start_time + timedelta(seconds=np.random.randint(0, 30*24*3600))
        pattern = np.random.choice(fraud_patterns)
        if pattern == 'high_amount':
            amount = float(np.round(np.random.uniform(3000, 15000), 2))
            country = np.random.choice(['US', 'RU', 'CN'])
            ev = 'payment'
        elif pattern == 'rare_country':
            amount = float(np.round(np.random.uniform(500, 3000), 2))
            country = np.random.choice(['Unknown', 'BR', 'IN'])
            ev = 'transfer'
        elif pattern == 'rapid_burst':
            amount = float(np.round(np.random.uniform(50, 500), 2))
            country = np.random.choice(countries)
            ev = 'payment'
        else:  # night_event
            ts = ts.replace(hour=np.random.randint(1, 5))
            amount = float(np.round(np.random.uniform(1000, 5000), 2))
            country = np.random.choice(countries)
            ev = 'transfer'
        
        records.append({
            'timestamp': ts,
            'user_id': np.random.choice(user_ids),
            'ip': gen_ip(),
            'amount': amount,
            'country': country,
            'event_type': ev,
            'is_fraud': 1
        })
    
    df = pd.DataFrame(records)
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    return df

df = generate_logs()
print(f'Всего записей: {len(df)}')
print(f'Мошеннических: {df["is_fraud"].sum()} ({df["is_fraud"].mean()*100:.2f}%)')
print(f'Нормальных:    {(df["is_fraud"]==0).sum()} ({(df["is_fraud"]==0).mean()*100:.2f}%)')

## Базовый анализ данных

Изучим распределение данных перед построением правил и моделей.

In [ ]:
# Просмотр первых строк
print('=== Первые 5 строк ===')
display(df.head())

print('\n=== Описание числовых признаков ===')
display(df.describe())

print('\n=== Типы событий ===')
print(df['event_type'].value_counts())

print('\n=== Транзакции по странам ===')
print(df['country'].value_counts())

In [ ]:
# Визуализация распределения сумм
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Гистограмма сумм (log scale)
axes[0, 0].hist(df[df['is_fraud']==0]['amount'], bins=50, alpha=0.6, label='Normal', color='steelblue')
axes[0, 0].hist(df[df['is_fraud']==1]['amount'], bins=50, alpha=0.6, label='Fraud', color='crimson')
axes[0, 0].set_xscale('log')
axes[0, 0].set_xlabel('Amount (log)')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('Distribution of amounts')
axes[0, 0].legend()

# События по странам
country_counts = df['country'].value_counts()
axes[0, 1].bar(country_counts.index, country_counts.values, color='teal')
axes[0, 1].set_xlabel('Country')
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title('Transactions by country')
axes[0, 1].tick_params(axis='x', rotation=45)

# Доля фрода по странам
fraud_by_country = df.groupby('country')['is_fraud'].mean().sort_values(ascending=False)
axes[1, 0].bar(fraud_by_country.index, fraud_by_country.values * 100, color='darkorange')
axes[1, 0].set_xlabel('Country')
axes[1, 0].set_ylabel('Fraud rate, %')
axes[1, 0].set_title('Fraud rate by country')
axes[1, 0].tick_params(axis='x', rotation=45)

# Доля фрода по типам событий
fraud_by_ev = df.groupby('event_type')['is_fraud'].mean().sort_values(ascending=False)
axes[1, 1].bar(fraud_by_ev.index, fraud_by_ev.values * 100, color='purple')
axes[1, 1].set_xlabel('Event type')
axes[1, 1].set_ylabel('Fraud rate, %')
axes[1, 1].set_title('Fraud rate by event type')

plt.tight_layout()
plt.show()

In [ ]:
# Корреляция между признаками и fraud
# Добавим час события как признак
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek

print('Средняя сумма для фрода vs норма:')
print(df.groupby('is_fraud')['amount'].agg(['mean', 'median', 'std']))

print('\nЧас событий (фрод vs норма):')
print(df.groupby('is_fraud')['hour'].agg(['mean', 'std']))

print('\nДоля ночных событий (1-5 часов):')
df['is_night'] = ((df['hour'] >= 1) & (df['hour'] <= 5)).astype(int)
print(df.groupby('is_fraud')['is_night'].mean())

## Правила поиска подозрительных событий (rule-based)

Перед ML попробуем простой подход: правила на основе порогов. Это то, с чего обычно начинают в Splunk - **SPL-правила**.

Правила:
1. Сумма > 5000 -> подозрительно
2. Страна в ['Unknown', 'BR'] -> +1 балл риска
3. Ночное событие (1-5 утра) -> +1 балл
4. >= 5 событий от одного пользователя за час -> +1 балл

Если суммарный балл >= 2 -> флаг fraud.

In [ ]:
# Rule-based детекция мошенничества

# Глобальные пороги правил (можно менять через виджеты далее)
RULES_CONFIG = {
    'amount_threshold': 5000,
    'risky_countries': ['Unknown', 'BR'],
    'night_hours': (1, 5),
    'rapid_burst_count': 5,
    'rapid_burst_window_min': 60,
    'fraud_score_threshold': 2
}

def apply_rules(df_in, config=None):
    # Применение rule-based правил к датафрейму
    cfg = config if config else RULES_CONFIG
    df_eval = df_in.copy()
    
    # Признак часа
    if 'hour' not in df_eval.columns:
        df_eval['hour'] = df_eval['timestamp'].dt.hour
    
    # Балл 1: превышение суммы
    df_eval['score_amount'] = (df_eval['amount'] > cfg['amount_threshold']).astype(int)
    
    # Балл 2: рискованная страна
    df_eval['score_country'] = df_eval['country'].isin(cfg['risky_countries']).astype(int)
    
    # Балл 3: ночное событие
    h_start, h_end = cfg['night_hours']
    df_eval['score_night'] = ((df_eval['hour'] >= h_start) & (df_eval['hour'] <= h_end)).astype(int)
    
    # Балл 4: rapid burst (>= N событий от пользователя за окно)
    df_eval = df_eval.sort_values(['user_id', 'timestamp'])
    df_eval['burst_count'] = 0
    window_sec = cfg['rapid_burst_window_min'] * 60
    for uid, group in df_eval.groupby('user_id'):
        ts_list = group['timestamp'].tolist()
        idx_list = group.index.tolist()
        for i, ts in enumerate(ts_list):
            cnt = sum(1 for t in ts_list if 0 <= (t - ts).total_seconds() <= window_sec)
            df_eval.loc[idx_list[i], 'burst_count'] = cnt
    df_eval['score_burst'] = (df_eval['burst_count'] >= cfg['rapid_burst_count']).astype(int)
    
    # Итоговый балл и флаг
    df_eval['risk_score'] = (df_eval['score_amount'] + df_eval['score_country'] + 
                              df_eval['score_night'] + df_eval['score_burst'])
    df_eval['predicted_fraud'] = (df_eval['risk_score'] >= cfg['fraud_score_threshold']).astype(int)
    
    return df_eval

# Применяем правила
df_rules = apply_rules(df)
print('Применены правила. Пример предсказаний:')
display(df_rules[['timestamp', 'user_id', 'amount', 'country', 'is_fraud', 'risk_score', 'predicted_fraud']].head(10))

In [ ]:
# Оценка качества правил: precision / recall / F1
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

y_true = df_rules['is_fraud']
y_pred = df_rules['predicted_fraud']

print('=== Метрики rule-based ===')
print(f'Accuracy:  {accuracy_score(y_true, y_pred):.4f}')
print(f'Precision: {precision_score(y_true, y_pred):.4f}')
print(f'Recall:    {recall_score(y_true, y_pred):.4f}')
print(f'F1-score:  {f1_score(y_true, y_pred):.4f}')

print('\n=== Confusion matrix ===')
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Fraud'], yticklabels=['Normal', 'Fraud'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion matrix: rule-based')
plt.show()

## ML-модель: RandomForest

Теперь обучим модель машинного обучения. Используем **RandomForestClassifier** - устойчив к переобучению и хорошо работает с категориальными признаками после One-Hot Encoding.

Шаги:
1. Подготовка признаков (One-Hot Encoding для country/event_type)
2. Train/test split (80/20)
3. Обучение RandomForest
4. Оценка через classification_report

In [ ]:
# Подготовка признаков для ML

# Копируем и удаляем утечки целевой переменной
df_ml = df_rules.copy()

# Признаки для модели
feature_cols = ['amount', 'country', 'event_type', 'hour', 'day_of_week',
                'is_night', 'burst_count', 'score_amount', 'score_country',
                'score_night', 'score_burst']

X = df_ml[feature_cols].copy()
y = df_ml['is_fraud'].values

# One-Hot Encoding для категориальных признаков
categorical = ['country', 'event_type']
numeric = [c for c in feature_cols if c not in categorical]

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical),
        ('num', 'passthrough', numeric)
    ]
)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train)} (fraud: {y_train.sum()})')
print(f'Test:  {len(X_test)} (fraud: {y_test.sum()})')

In [ ]:
# Обучение RandomForest
MODEL_PARAMS = {
    'n_estimators': 100,
    'max_depth': 10,
    'random_state': 42,
    'class_weight': 'balanced',
    'n_jobs': -1
}

# Pipeline: preprocessing + RandomForest
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(**MODEL_PARAMS))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print('=== Classification report ===')
print(classification_report(y_test, y_pred, target_names=['Normal', 'Fraud']))

In [ ]:
# Сравнение rule-based vs ML
rules_pred_test = df_rules.loc[X_test.index, 'predicted_fraud'].values

print('=== Сравнение подходов на тестовой выборке ===')
print(f"{'Метрика':<12}{'Rule-based':<15}{'ML (RF)':<15}")
print('-' * 42)
for metric_fn, name in [(precision_score, 'Precision'), (recall_score, 'Recall'), 
                        (f1_score, 'F1-score'), (accuracy_score, 'Accuracy')]:
    r_val = metric_fn(y_test, rules_pred_test)
    m_val = metric_fn(y_test, y_pred)
    print(f'{name:<12}{r_val:<15.4f}{m_val:<15.4f}')

# Confusion matrix для ML-модели
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
cm_rules = confusion_matrix(y_test, rules_pred_test)
cm_ml = confusion_matrix(y_test, y_pred)
sns.heatmap(cm_rules, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Normal', 'Fraud'], yticklabels=['Normal', 'Fraud'])
axes[0].set_title('Rule-based')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
sns.heatmap(cm_ml, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=['Normal', 'Fraud'], yticklabels=['Normal', 'Fraud'])
axes[1].set_title('ML (RandomForest)')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
# Важность признаков в RandomForest
rf_model = pipeline.named_steps['classifier']
encoder = pipeline.named_steps['preprocessor'].named_transformers_['cat']
cat_features = encoder.get_feature_names_out(categorical).tolist()
all_features = cat_features + numeric

importances = rf_model.feature_importances_
feat_df = pd.DataFrame({'feature': all_features, 'importance': importances})
feat_df = feat_df.sort_values('importance', ascending=True).tail(15)

plt.figure(figsize=(10, 6))
plt.barh(feat_df['feature'], feat_df['importance'], color='darkcyan')
plt.xlabel('Importance')
plt.title('Top-15 feature importances (RandomForest)')
plt.tight_layout()
plt.show()

## Имитация работы Splunk: SPL-запросы

В реальном Splunk аналитики пишут запросы на языке **SPL**. Покажем несколько типичных запросов для fraud detection и их эквивалент на pandas.

### Запрос 1: Все транзакции с суммой > 5000

**SPL:**
```
index=transactions amount>5000
| table timestamp user_id amount country event_type is_fraud
| sort -amount
```

**Pandas-эквивалент:**

In [ ]:
# SPL -> pandas: amount > 5000
spl_result_1 = df[df['amount'] > 5000][
    ['timestamp', 'user_id', 'amount', 'country', 'event_type', 'is_fraud']
].sort_values('amount', ascending=False)
print(f'Найдено {len(spl_result_1)} транзакций с amount > 5000')
display(spl_result_1.head(10))

### Запрос 2: Топ-10 пользователей по числу мошеннических событий

**SPL:**
```
index=transactions is_fraud=1
| stats count as fraud_count by user_id
| sort -fraud_count
| head 10
```

**Pandas-эквивалент:**

In [ ]:
# SPL -> pandas: топ-10 пользователей по фроду
spl_result_2 = (df[df['is_fraud']==1]
                .groupby('user_id')
                .size()
                .reset_index(name='fraud_count')
                .sort_values('fraud_count', ascending=False)
                .head(10))
display(spl_result_2)

### Запрос 3: Доля фрода по странам (за последние 7 дней)

**SPL:**
```
index=transactions earliest=-7d
| stats count as total, sum(is_fraud) as frauds by country
| eval fraud_rate = round(frauds/total * 100, 2)
| sort -fraud_rate
```

**Pandas-эквивалент:**

In [ ]:
# SPL -> pandas: доля фрода по странам
spl_result_3 = (df.groupby('country')
                .agg(total=('is_fraud', 'count'),
                     frauds=('is_fraud', 'sum'))
                .reset_index())
spl_result_3['fraud_rate'] = (spl_result_3['frauds'] / spl_result_3['total'] * 100).round(2)
spl_result_3 = spl_result_3.sort_values('fraud_rate', ascending=False)
display(spl_result_3)

### Запрос 4: Обнаружение compromised account (несколько failed logins + смена IP)

**SPL:**
```
index=auth event_type=login
| stats count as attempts, dc(ip) as distinct_ips by user_id
| where attempts > 3 AND distinct_ips > 2
| sort -attempts
```

**Pandas-эквивалент:**

In [ ]:
# SPL -> pandas: compromised account detection
auth_df = df[df['event_type']=='login']
spl_result_4 = (auth_df.groupby('user_id')
                .agg(attempts=('ip', 'count'),
                     distinct_ips=('ip', 'nunique'))
                .reset_index())
spl_result_4 = spl_result_4[
    (spl_result_4['attempts'] > 3) & (spl_result_4['distinct_ips'] > 2)
].sort_values('attempts', ascending=False)
print(f'Подозрительные аккаунты: {len(spl_result_4)}')
display(spl_result_4.head(10))

## Реальные кейсы fraud detection

### Кейс 1: Compromised account (взломанный аккаунт)

**Сценарий**: злоумышленник получил доступ к аккаунту пользователя. Характерные признаки:
- Много failed login attempts за короткое время
- Резкая смена IP-адреса или геолокации
- Необычное время активности (ночью)
- Покупки/переводы, нетипичные для пользователя

### Кейс 2: Bonus program fraud (фрод бонусной программы)

**Сценарий**: мошенники регистрируют множество аккаунтов для получения приветственных бонусов. Признаки:
- Один IP на много user_id
- Один device fingerprint
- Быстрое обналичивание бонусов
- Похожие имена/почты

In [ ]:
# Кейс 1: Compromised account - поиск пользователей с аномальным поведением

# Считаем для каждого пользователя число событий, число уникальных IP, среднюю сумму
user_stats = (df.groupby('user_id')
               .agg(events_count=('event_id', 'count') if 'event_id' in df.columns else ('timestamp', 'count'),
                    unique_ips=('ip', 'nunique'),
                    avg_amount=('amount', 'mean'),
                    fraud_events=('is_fraud', 'sum'))
               .reset_index())

# Флаг подозрительного аккаунта: > 1 IP и хотя бы 1 мошенническое событие
suspicious = user_stats[(user_stats['unique_ips'] > 1) & (user_stats['fraud_events'] > 0)]
print(f'Подозрительные аккаунты (compromised): {len(suspicious)}')
display(suspicious.sort_values('fraud_events', ascending=False).head(10))

In [ ]:
# Кейс 2: Bonus program fraud - поиск IP с большим числом user_id
ip_user_counts = (df.groupby('ip')['user_id']
                    .nunique()
                    .reset_index(name='distinct_users'))
ip_user_counts = ip_user_counts[ip_user_counts['distinct_users'] >= 3].sort_values('distinct_users', ascending=False)

print(f'IP-адреса с >= 3 пользователями (potential bonus fraud): {len(ip_user_counts)}')
display(ip_user_counts.head(10))

# Если IP используется многими пользователями - это сигнал
risky_ips = set(ip_user_counts['ip'].head(20).tolist())
print(f'\nТоп-20 рискованных IP добавлены в watchlist')
print(f'Пример: {list(risky_ips)[:3]}')

## FastAPI-сервер для детекции мошенничества

Завернём наши правила и ML-модель в **FastAPI**-приложение. Эндпоинты:

- `POST /detect` - проверить одну транзакцию
- `POST /batch` - пакетная проверка
- `GET /rules` - получить текущие правила
- `PUT /rules` - обновить правила
- `GET /health` - проверка здоровья

Для доступа извне Colab используем **ngrok**.

In [ ]:
# Подготовка FastAPI-сервера
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List, Optional
import uvicorn
from pyngrok import ngrok

# Модель входных данных
class Transaction(BaseModel):
    user_id: str
    amount: float
    country: str
    event_type: str
    ip: str = ""
    hour: Optional[int] = None

class BatchRequest(BaseModel):
    transactions: List[Transaction]

class RulesUpdate(BaseModel):
    amount_threshold: Optional[int] = None
    risky_countries: Optional[List[str]] = None
    fraud_score_threshold: Optional[int] = None

# Глобальное состояние правил
current_rules = RULES_CONFIG.copy()

# Создаём приложение FastAPI
app = FastAPI(title="Splunk-style Fraud Detection API", version="1.0")

@app.get('/health')
def health():
    # Проверка здоровья API
    return {'status': 'ok', 'model_loaded': pipeline is not None}

@app.get('/rules')
def get_rules():
    # Получить текущие правила
    return current_rules

@app.put('/rules')
def update_rules(rules: RulesUpdate):
    # Обновить правила
    global current_rules
    if rules.amount_threshold is not None:
        current_rules['amount_threshold'] = rules.amount_threshold
    if rules.risky_countries is not None:
        current_rules['risky_countries'] = rules.risky_countries
    if rules.fraud_score_threshold is not None:
        current_rules['fraud_score_threshold'] = rules.fraud_score_threshold
    return {'status': 'updated', 'rules': current_rules}

def score_transaction(tx: Transaction) -> dict:
    # Оценка одной транзакции: rule-based + ML
    # Rule-based скоринг
    risk_score = 0.0
    reasons = []
    
    if tx.amount > current_rules['amount_threshold']:
        risk_score += 0.4
        reasons.append(f"Amount {tx.amount} exceeds threshold {current_rules['amount_threshold']}")
    
    if tx.country in current_rules['risky_countries']:
        risk_score += 0.3
        reasons.append(f"Country {tx.country} is risky")
    
    h = tx.hour if tx.hour is not None else datetime.now().hour
    h_start, h_end = current_rules['night_hours']
    if h_start <= h <= h_end:
        risk_score += 0.2
        reasons.append(f"Night event (hour={h})")
    
    # ML-предсказание
    ml_proba = 0.0
    try:
        feat_dict = {
            'amount': tx.amount,
            'country': tx.country,
            'event_type': tx.event_type,
            'hour': h,
            'day_of_week': datetime.now().weekday(),
            'is_night': int(h_start <= h <= h_end),
            'burst_count': 1,
            'score_amount': int(tx.amount > current_rules['amount_threshold']),
            'score_country': int(tx.country in current_rules['risky_countries']),
            'score_night': int(h_start <= h <= h_end),
            'score_burst': 0
        }
        X_single = pd.DataFrame([feat_dict])
        ml_proba = float(pipeline.predict_proba(X_single)[0, 1])
    except Exception as e:
        reasons.append(f"ML prediction failed: {e}")
    
    # Комбинированный риск
    combined_risk = 0.5 * risk_score + 0.5 * ml_proba
    is_fraud = combined_risk > 0.5
    
    return {
        'is_fraud': is_fraud,
        'risk_score': round(combined_risk, 4),
        'rule_score': round(risk_score, 4),
        'ml_probability': round(ml_proba, 4),
        'reasons': reasons
    }

@app.post('/detect')
def detect_fraud(tx: Transaction):
    # Проверить одну транзакцию
    return score_transaction(tx)

@app.post('/batch')
def detect_batch(batch: BatchRequest):
    # Пакетная проверка транзакций
    results = []
    for tx in batch.transactions:
        results.append({'user_id': tx.user_id, **score_transaction(tx)})
    summary = {
        'total': len(results),
        'fraud_detected': sum(1 for r in results if r['is_fraud']),
        'results': results
    }
    return summary

print('FastAPI приложение создано.')

In [ ]:
# Запуск FastAPI сервера через ngrok
# Внимание: эта ячейка блокирует выполнение. Остановите её, когда закончите тестирование.

# Раскомментируйте следующие 2 строки, чтобы запустить сервер:
# public_url = ngrok.connect(8000)
# print(f'API доступно по адресу: {public_url}')
# print(f'Документация: {public_url}/docs')
# uvicorn.run(app, host='0.0.0.0', port=8000)

# Для тестирования без блокировки можно запустить в отдельном потоке:
def run_server():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='error')

# Запуск в фоне
public_url = ngrok.connect(8000)
print(f'API доступно: {public_url}')
print(f'Swagger UI:   {public_url}/docs')
print(f'Health:       {public_url}/health')

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
import time; time.sleep(3)
print('Сервер запущен в фоне.')

In [ ]:
# Тестирование API: примеры запросов
import requests

base_url = str(public_url)

# 1. Health check
r = requests.get(f'{base_url}/health')
print('Health:', r.json())

# 2. Получить правила
r = requests.get(f'{base_url}/rules')
print('\nCurrent rules:', r.json())

# 3. Проверить одну транзакцию (нормальную)
normal_tx = {
    'user_id': 'user_0042',
    'amount': 150.0,
    'country': 'US',
    'event_type': 'payment',
    'ip': '192.168.1.10',
    'hour': 14
}
r = requests.post(f'{base_url}/detect', json=normal_tx)
print('\nNormal transaction:', r.json())

# 4. Проверить подозрительную транзакцию
suspicious_tx = {
    'user_id': 'user_0999',
    'amount': 8000.0,
    'country': 'Unknown',
    'event_type': 'transfer',
    'ip': '10.0.0.5',
    'hour': 3
}
r = requests.post(f'{base_url}/detect', json=suspicious_tx)
print('\nSuspicious transaction:', r.json())

# 5. Пакетная проверка
batch = {'transactions': [normal_tx, suspicious_tx, normal_tx]}
r = requests.post(f'{base_url}/batch', json=batch)
print('\nBatch result:', r.json())

## Интерактивные виджеты

Настроим параметры детекции через **ipywidgets**:
- Слайдер порога суммы
- Dropdown страны
- Слайдеры для параметров ML (n_estimators, max_depth)
- Кнопка переобучения модели
- Кнопка применения правил

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- Виджеты для правил ---
amount_slider = widgets.IntSlider(
    value=RULES_CONFIG['amount_threshold'],
    min=1000, max=20000, step=500,
    description='Порог суммы:',
    style={'description_width': 'initial'}
)

country_dropdown = widgets.Dropdown(
    options=['All', 'US', 'RU', 'CN', 'DE', 'FR', 'UK', 'BR', 'IN', 'JP', 'Unknown'],
    value='All',
    description='Страна фильтр:',
    style={'description_width': 'initial'}
)

fraud_threshold_slider = widgets.IntSlider(
    value=RULES_CONFIG['fraud_score_threshold'],
    min=1, max=4, step=1,
    description='Порог fraud-score:',
    style={'description_width': 'initial'}
)

# --- Виджеты для ML ---
n_estimators_slider = widgets.IntSlider(
    value=100, min=10, max=500, step=10,
    description='n_estimators:',
    style={'description_width': 'initial'}
)

max_depth_slider = widgets.IntSlider(
    value=10, min=2, max=50, step=1,
    description='max_depth:',
    style={'description_width': 'initial'}
)

# --- Кнопки ---
apply_rules_btn = widgets.Button(
    description='Применить правила',
    button_style='primary',
    icon='check'
)

retrain_btn = widgets.Button(
    description='Переобучить модель',
    button_style='warning',
    icon='refresh'
)

output = widgets.Output()

def on_apply_rules(b):
    # Обновить глобальные правила и пересчитать метрики
    global current_rules
    current_rules['amount_threshold'] = amount_slider.value
    current_rules['fraud_score_threshold'] = fraud_threshold_slider.value
    
    with output:
        clear_output()
        print(f'Правила обновлены:')
        print(f'  amount_threshold = {current_rules["amount_threshold"]}')
        print(f'  fraud_score_threshold = {current_rules["fraud_score_threshold"]}')
        
        # Пересчёт rule-based
        df_new = apply_rules(df, current_rules)
        y_t = df_new['is_fraud']
        y_p = df_new['predicted_fraud']
        print('\nМетрики rule-based после обновления:')
        print(f'  Precision: {precision_score(y_t, y_p):.4f}')
        print(f'  Recall:    {recall_score(y_t, y_p):.4f}')
        print(f'  F1-score:  {f1_score(y_t, y_p):.4f}')
        
        # Фильтр по стране
        if country_dropdown.value != 'All':
            df_f = df_new[df_new['country'] == country_dropdown.value]
            print(f'\nОтфильтровано по стране {country_dropdown.value}: {len(df_f)} записей')
            print(f'  Из них фрод: {df_f["is_fraud"].sum()}')

def on_retrain(b):
    # Переобучить модель с новыми параметрами
    global pipeline, MODEL_PARAMS
    MODEL_PARAMS['n_estimators'] = n_estimators_slider.value
    MODEL_PARAMS['max_depth'] = max_depth_slider.value
    
    with output:
        clear_output()
        print(f'Переобучение с параметрами: {MODEL_PARAMS}')
        pipeline = Pipeline([
            ('preprocessor', preprocessor),
            ('classifier', RandomForestClassifier(**MODEL_PARAMS))
        ])
        pipeline.fit(X_train, y_train)
        y_pred_new = pipeline.predict(X_test)
        print('\nМетрики ML после переобучения:')
        print(f'  Precision: {precision_score(y_test, y_pred_new):.4f}')
        print(f'  Recall:    {recall_score(y_test, y_pred_new):.4f}')
        print(f'  F1-score:  {f1_score(y_test, y_pred_new):.4f}')
        print(f'  Accuracy:  {accuracy_score(y_test, y_pred_new):.4f}')

apply_rules_btn.on_click(on_apply_rules)
retrain_btn.on_click(on_retrain)

print('=== Виджеты для правил ===')
display(amount_slider, country_dropdown, fraud_threshold_slider, apply_rules_btn)
print('\n=== Виджеты для ML-модели ===')
display(n_estimators_slider, max_depth_slider, retrain_btn)
print('\n=== Вывод ===')
display(output)

In [ ]:
# Виджет для интерактивной проверки одной транзакции
tx_amount = widgets.FloatSlider(value=1000, min=0, max=20000, step=100,
                                 description='Сумма:', style={'description_width': 'initial'})
tx_country = widgets.Dropdown(options=['US','RU','CN','DE','FR','UK','BR','IN','JP','Unknown'],
                               value='US', description='Страна:', style={'description_width': 'initial'})
tx_event = widgets.Dropdown(options=['login','payment','transfer','refund'],
                             value='payment', description='Событие:', style={'description_width': 'initial'})
tx_hour = widgets.IntSlider(value=12, min=0, max=23, step=1,
                             description='Час:', style={'description_width': 'initial'})
check_btn = widgets.Button(description='Проверить транзакцию', button_style='success', icon='search')
tx_output = widgets.Output()

def on_check(b):
    # Отправить транзакцию в API и показать результат
    tx = {
        'user_id': 'widget_user',
        'amount': tx_amount.value,
        'country': tx_country.value,
        'event_type': tx_event.value,
        'ip': '127.0.0.1',
        'hour': tx_hour.value
    }
    with tx_output:
        clear_output()
        print(f'Транзакция: amount={tx["amount"]}, country={tx["country"]}, event={tx["event_type"]}, hour={tx["hour"]}')
        try:
            r = requests.post(f'{str(public_url)}/detect', json=tx)
            result = r.json()
            print(f'\nРезультат:')
            print(f'  is_fraud:       {result["is_fraud"]}')
            print(f'  risk_score:     {result["risk_score"]}')
            print(f'  rule_score:     {result["rule_score"]}')
            print(f'  ml_probability: {result["ml_probability"]}')
            if result['reasons']:
                print(f'  Причины:')
                for rsn in result['reasons']:
                    print(f'    - {rsn}')
        except Exception as e:
            print(f'Ошибка: {e}. Запустите FastAPI-сервер выше.')

check_btn.on_click(on_check)
display(tx_amount, tx_country, tx_event, tx_hour, check_btn, tx_output)

## Практические задания

Выполните задания для закрепления материала.

### Задание 1: Изменить порог суммы
Используя виджет `amount_slider`, установите порог 3000 и примените правила. Сравните precision и recall с исходным порогом 5000. Сделайте выводы.

### Задание 2: Добавить новый признак
Добавьте в `feature_cols` признак `is_weekend` (1 если суббота/воскресенье). Переобучите модель. Изменилось ли качество?

**Подсказка**:
```python
df_ml['is_weekend'] = df_ml['day_of_week'].isin([5, 6]).astype(int)
```

### Задание 3: Написать собственное правило
Добавьте в функцию `apply_rules` новое правило: флаг если пользователь делает > 3 переводов (transfer) за 10 минут. Реализуйте логику по аналогии с `score_burst`.

### Задание 4: Сравнить модели
Замените RandomForest на GradientBoostingClassifier или LogisticRegression. Сравните F1-метрики.

### Задание 5: Создать эндпоинт `/explain`
Добавьте в FastAPI эндпоинт, который возвращает не только риск, но и SHAP-значения топ-3 признаков, повлиявших на решение.

## Заключение

### Что мы сделали

1. Сгенерировали синтетические логи транзакций в стиле Splunk
2. Реализовали **rule-based** детекцию с порогами и балльной системой
3. Обучили **RandomForest** с One-Hot Encoding для категориальных признаков
4. Показали **SPL-запросы** и их эквиваленты на pandas
5. Разобрали реальные кейсы: compromised account и bonus program fraud
6. Завернули всё в **FastAPI** с эндпоинтами `/detect`, `/batch`, `/rules`
7. Настроили **ipywidgets** для интерактивной настройки параметров

### Перенос в реальную инфраструктуру

Для продакшена рекомендуется:

- **Splunk Enterprise** + **Splunk Machine Learning Toolkit** - встроенная интеграция с sklearn-моделями
- **Apache Kafka** для стриминга событий
- **Elasticsearch + Kibana** как open-source альтернатива Splunk
- **MLflow** для версионирования моделей
- **Prometheus + Grafana** для мониторинга качества детекции

### Полезные ссылки

- [Splunk Documentation](https://docs.splunk.com/)
- [Splunk ML Toolkit](https://docs.splunk.com/Documentation/MLApp)
- [SPL Search Reference](https://docs.splunk.com/Documentation/Search)
- [scikit-learn RandomForest](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)
- [FastAPI Documentation](https://fastapi.tiangolo.com/)
- [ngrok - Quickstart](https://ngrok.com/docs)

---

**Следующий шаг**: попробуйте подключить этот ноутбук к реальным логам (например, через Splunk REST API или чтение JSON-логов из файла) и оцените качество модели на ваших данных.